In [2]:
import os
import pandas as pd
import numpy as np
import re
from pathlib import Path

initialDB_path = Path('../data/initialDB_sequences.csv')
initialDB = pd.read_csv(initialDB_path, sep=',', header=0, comment='#', names=[
    'gene_id', 'transcript_id', '5_UTR', 'CDS', '3_UTR', '5_UTR_len', 'CDS_len', '3_UTR_len'
])

df = initialDB.copy()
df.head()

,gene_id,transcript_id,5_UTR,CDS,3_UTR,5_UTR_len,CDS_len,3_UTR_len
0,ENSG00000186092.7,ENST00000641515.2,CCCAGATCTCTTCAGTTTTTATGCCTCATTCTGTGAAAATTGCTGT...,ATGAAGAAGGTAACTGCAGAGGCTATTTCCTGGAATGAATCAACGA...,NaN,60,978,0
1,ENSG00000284733.2,ENST00000426406.4,NaN,ATGGATGGAGAGAATCACTCAGTGGTATCTGAGTTTTTGTTTCTGG...,TAA,0,936,3
2,ENSG00000284662.2,ENST00000332831.5,NaN,ATGGATGGAGAGAATCACTCAGTGGTATCTGAGTTTTTGTTTCTGG...,TAA,0,936,3
3,ENSG00000187634.14,ENST00000616016.5,GGCGGCGGAGTCTCCCAAGTCCCCGCCGGGCGGGCGCGCGCCAGTG...,ATGCCGGCGGTCAAGAAGGAGTTCCCGGGCCGCGAGGACCTGGCCC...,NaN,509,2532,0
4,ENSG00000188976.12,ENST00000327044.7,GCTTCGGGTTGGTGTC,ATGGCAGCTGCGGGGAGCCGCAAGAGGCGCCTGGCGGAGCTGACGG...,TGAGGCAGCCCATCTGGGGGGCCTGTAGGGGCTGCCGGGCTGGTGG...,16,2247,494


In [13]:
def gc_content(seq):
    if not isinstance(seq, str):
        return np.nan
    seq = seq.strip().upper()
    if not seq:
        return np.nan
    gc = seq.count('G') + seq.count('C')
    return gc / len(seq)

df = initialDB.copy()
df['5_UTR_GC'] = df['5_UTR'].fillna('').map(gc_content)
df['CDS_GC'] = df['CDS'].fillna('').map(gc_content)
df['3_UTR_GC'] = df['3_UTR'].fillna('').map(gc_content)
df['global_GC'] = df[['5_UTR', 'CDS', '3_UTR']].fillna('').agg(''.join, axis=1).map(gc_content)

df.head()

,gene_id,transcript_id,5_UTR,CDS,3_UTR,5_UTR_len,CDS_len,3_UTR_len,5_UTR_GC,CDS_GC,3_UTR_GC,global_GC
0,ENSG00000186092.7,ENST00000641515.2,CCCAGATCTCTTCAGTTTTTATGCCTCATTCTGTGAAAATTGCTGT...,ATGAAGAAGGTAACTGCAGAGGCTATTTCCTGGAATGAATCAACGA...,NaN,60,978,0,0.40000,0.426380,NaN,0.424855
1,ENSG00000284733.2,ENST00000426406.4,NaN,ATGGATGGAGAGAATCACTCAGTGGTATCTGAGTTTTTGTTTCTGG...,TAA,0,936,3,NaN,0.461538,0.000000,0.460064
2,ENSG00000284662.2,ENST00000332831.5,NaN,ATGGATGGAGAGAATCACTCAGTGGTATCTGAGTTTTTGTTTCTGG...,TAA,0,936,3,NaN,0.461538,0.000000,0.460064
3,ENSG00000187634.14,ENST00000616016.5,GGCGGCGGAGTCTCCCAAGTCCCCGCCGGGCGGGCGCGCGCCAGTG...,ATGCCGGCGGTCAAGAAGGAGTTCCCGGGCCGCGAGGACCTGGCCC...,NaN,509,2532,0,0.80943,0.701422,NaN,0.719500
4,ENSG00000188976.12,ENST00000327044.7,GCTTCGGGTTGGTGTC,ATGGCAGCTGCGGGGAGCCGCAAGAGGCGCCTGGCGGAGCTGACGG...,TGAGGCAGCCCATCTGGGGGGCCTGTAGGGGCTGCCGGGCTGGTGG...,16,2247,494,0.62500,0.603471,0.572874,0.598114


In [14]:
motifs = {
"ATTTA": r"A[TU]{3}A",
"WTTTW": r"[ACGTU][TU]{3}[ACGTU]",
"WWTTTWW": r"[ACGTU]{2}[TU]{3}[ACGTU]{2}",
"WWWTTTWWW": r"[ACGTU]{3}[TU]{3}[ACGTU]{3}",
"WWWWTTTWWWW": r"[ACGTU]{4}[TU]{3}[ACGTU]{4}",
"WWWWWTTTWWWWW": r"[ACGTU]{5}[TU]{3}[ACGTU]{5}",
"TTTGTTT": r"[TU]{3}G[TU]{3}",
"GTTTG": r"G[TU]{3}G",
"AWTAAA": r"A[ACGTU][TU]AAA"
}

def scan_motifs(df, sequence_col, count_prefix):
    """
    Scans a specific column for motifs and adds a total count column.
    Returns a detailed breakdown DataFrame for those specific motifs.
    """
    total_col_name = f"{count_prefix}_AREs"
    df[total_col_name] = 0
    
    motif_counts_dict = {
        'gene_id': df['gene_id'],
        'transcript_id': df['transcript_id'],
        f'{count_prefix}_sequence': df[sequence_col]
    }
    
    for name, pattern in motifs.items():
        counts = df[sequence_col].str.findall(pattern, flags=re.IGNORECASE).str.len().fillna(0).astype(int)
        
        df[total_col_name] += counts
        
        motif_counts_dict[f"{count_prefix}_{name}_count"] = counts
    
    detailed_df = pd.DataFrame(motif_counts_dict)
    return df, detailed_df

df, AREs_5_df = scan_motifs(df, '5_UTR', '5')

df, AREs_3_df = scan_motifs(df, '3_UTR', '3')

df.head(10)

,gene_id,transcript_id,5_UTR,CDS,3_UTR,5_UTR_len,CDS_len,3_UTR_len,5_UTR_GC,CDS_GC,3_UTR_GC,global_GC,5_AREs,3_AREs
0,ENSG00000186092.7,ENST00000641515.2,CCCAGATCTCTTCAGTTTTTATGCCTCATTCTGTGAAAATTGCTGT...,ATGAAGAAGGTAACTGCAGAGGCTATTTCCTGGAATGAATCAACGA...,NaN,60,978,0,0.400000,0.426380,NaN,0.424855,5,0
1,ENSG00000284733.2,ENST00000426406.4,NaN,ATGGATGGAGAGAATCACTCAGTGGTATCTGAGTTTTTGTTTCTGG...,TAA,0,936,3,NaN,0.461538,0.000000,0.460064,0,0
2,ENSG00000284662.2,ENST00000332831.5,NaN,ATGGATGGAGAGAATCACTCAGTGGTATCTGAGTTTTTGTTTCTGG...,TAA,0,936,3,NaN,0.461538,0.000000,0.460064,0,0
3,ENSG00000187634.14,ENST00000616016.5,GGCGGCGGAGTCTCCCAAGTCCCCGCCGGGCGGGCGCGCGCCAGTG...,ATGCCGGCGGTCAAGAAGGAGTTCCCGGGCCGCGAGGACCTGGCCC...,NaN,509,2532,0,0.809430,0.701422,NaN,0.719500,0,0
4,ENSG00000188976.12,ENST00000327044.7,GCTTCGGGTTGGTGTC,ATGGCAGCTGCGGGGAGCCGCAAGAGGCGCCTGGCGGAGCTGACGG...,TGAGGCAGCCCATCTGGGGGGCCTGTAGGGGCTGCCGGGCTGGTGG...,16,2247,494,0.625000,0.603471,0.572874,0.598114,0,45
5,ENSG00000187961.16,ENST00000338591.8,GGGAGTGAGCGACACAGAGCGGGCCGCCACCGCCGAGCAGCCCTCC...,ATGCAGCCCCGCAGCGAGCGCCCGGCCGGCAGGACGCAGAGCCCGG...,NaN,110,1926,0,0.745455,0.663032,NaN,0.667485,0,0
6,ENSG00000187583.12,ENST00000379410.8,AGGAGGCTGTGGACAGGGACCCAGACTTGCCGACCTGTACGACTCT...,ATGGGGAACAGCCACTGTGTCCCTCAGGCCCCCAGGAGGCTCCGGG...,NaN,50,1833,0,0.640000,0.667758,NaN,0.667021,0,0
7,ENSG00000187642.11,ENST00000433179.4,AGACGCCCAGCTCGGCCGCCGGGACCCAGGGCATGGATGGAGCCCC...,ATGGAAAATTTCCAGTACAGCGTCCAGCTGAGTGACCAGGACTGGG...,TAGGAGCCAGGCCCGGGCCAGGGAGATGCAGGATGAGGAGACGACC...,173,2370,977,0.705202,0.660759,0.634596,0.655682,0,27
8,ENSG00000188290.12,ENST00000304952.11,GCGGGCCTGGAGCCGGGATCCGCCCTAGGGGCTCGGATCGCCGCGC...,ATGGCCGCAGACACGCCGGGGAAACCGAGCGCCTCGCCGATGGCAG...,TGAGGCTGTGGCCCTGAGACTGCATCGGAGGCGGCGCCCCGTTCTA...,124,663,98,0.838710,0.743590,0.571429,0.737853,0,11
9,ENSG00000187608.11,ENST00000649529.1,GGCGGCTGAGAGGCAGCGAACTCATCTTTGCCAGTACAGGAGCTTG...,ATGGGCTGGGACCTGACGGTGAAGATGCTGGCGGGCAACGAATTCC...,NaN,77,495,0,0.649351,0.658586,NaN,0.657343,5,0
